# Compatibilidad estética y aprendizaje métrico con Fashion-CLIP

Este notebook entrena una capa de proyección de 128 dimensiones sobre representaciones visuales de Fashion-CLIP. El extractor de características se mantiene fijo para conservar el conocimiento visual del modelo base, mientras que la capa densa aprende un espacio métrico ajustado a la coherencia entre prendas y a 12 estilos de vestimenta.

El flujo incluye:
1. Definición del modelo de proyección y la función de pérdida contrastiva (InfoNCE).
2. Procesamiento de los datos de entrenamiento y validación desde `sample_data/`.
3. Bucle de optimización conjunto (compatibilidad de pares y estilo).
4. Calibración de probabilidades mediante escalado de temperatura.
5. Evaluación con la prueba Fill-In-The-Blank (FITB) frente a distractores.
6. Exportación del archivo de pesos a Google Drive (`models/fashion_clip_adapter.pt`).


## 1. Entorno y configuración de rutas
Configura el acceso a Google Drive y define las rutas de lectura para `sample_data/` y escritura para `models/`.


In [ ]:
import os
import sys
import json
import random
from pathlib import Path

# Detección de entorno
IS_COLAB = Path("/content").exists()

if IS_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    
    possible_roots = [
        Path("/content/drive/MyDrive/MACHINE-BULLS"),
        Path("/content/drive/MyDrive/machine-bulls"),
        Path("/content/drive/My Drive/MACHINE-BULLS"),
    ]
    REPO_ROOT = Path("/content")
    for p in possible_roots:
        if p.exists():
            REPO_ROOT = p
            break
    print(f"Entorno Colab listo. Carpeta de trabajo: {REPO_ROOT}")
else:
    REPO_ROOT = Path.cwd().resolve()
    for parent in [REPO_ROOT, *REPO_ROOT.parents]:
        if (parent / "sample_data").exists():
            REPO_ROOT = parent
            break
    print(f"Entorno local listo. Carpeta de trabajo: {REPO_ROOT}")

SAMPLE_DIR = REPO_ROOT / "sample_data"
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Datos en: {SAMPLE_DIR} (disponible: {SAMPLE_DIR.exists()})")
print(f"Modelos en: {MODELS_DIR}")


## 2. Dependencias
Instala los paquetes necesarios en la sesión de Colab.


In [ ]:
if IS_COLAB:
    !pip install -q transformers dagshub mlflow pyyaml pillow
    print("Dependencias listas.")


## 3. Modelo y pérdida contrastiva
Define la red de proyección de 128 dimensiones (`AestheticProjectionHead`) y la función de pérdida InfoNCE simétrica para medir coherencia visual entre prendas.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

# Catálogo de 12 estilos de vestimenta
ESTETICAS = (
    "old money", "streetwear", "minimalist", "casual",
    "formal", "gorpcore", "dark academia", "y2k",
    "soft boy", "starboy", "grunge", "vintage",
)

class AestheticProjectionHead(nn.Module):
    """Proyecta representaciones de 512 dimensiones a un subespacio normalizado de 128 dimensiones."""

    def __init__(self,
        input_dim: int = 512,
        hidden_dim: int = 256,
        output_dim: int = 128,
        num_aesthetics: int = 12,
        dropout: float = 0.2,
    ):
        super().__init__()
        self.output_dim = output_dim

        # Capas de proyección con normalización L2
        self.projection = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, output_dim),
        )

        # Clasificador de estilo
        self.classifier = nn.Sequential(
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(output_dim, num_aesthetics),
        )

    def forward(self, x: torch.Tensor):
        raw_emb = self.projection(x)
        norm_emb = F.normalize(raw_emb, p=2, dim=-1)
        logits = self.classifier(norm_emb)
        return norm_emb, logits

    def compute_similarity(self, a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
        na = F.normalize(a, p=2, dim=-1)
        nb = F.normalize(b, p=2, dim=-1)
        return (na * nb).sum(dim=-1)


class ContrastiveCompatibilityLoss(nn.Module):
    """Pérdida contrastiva InfoNCE simétrica para pares de prendas."""

    def __init__(self, temperature: float = 0.07):
        super().__init__()
        self.temperature = temperature

    def forward(self, emb_anchor: torch.Tensor, emb_positive: torch.Tensor) -> torch.Tensor:
        batch_size = emb_anchor.shape[0]
        if batch_size <= 1:
            return torch.tensor(0.0, requires_grad=True, device=emb_anchor.device)

        sim = torch.matmul(emb_anchor, emb_positive.T) / self.temperature
        labels = torch.arange(batch_size, device=emb_anchor.device)

        loss_a = F.cross_entropy(sim, labels)
        loss_b = F.cross_entropy(sim.T, labels)
        return (loss_a + loss_b) / 2.0


print("Arquitectura del modelo y función de pérdida definidas.")


## 4. Conjunto de datos
Carga los pares de prendas desde los archivos JSONL en `sample_data/` y prepara los DataLoaders de PyTorch.


In [ ]:
class GarmentPairDataset(Dataset):
    """Carga pares de prendas y etiquetas de estilo desde archivos JSONL."""

    def __init__(self, jsonl_path: Path, embedding_dim: int = 512, seed: int = 42, max_samples: int = None):
        self.items = []
        if jsonl_path.exists():
            with open(jsonl_path, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if line:
                        self.items.append(json.loads(line))
        else:
            print(f"Aviso: No se encontró {jsonl_path}. Se usarán datos sintéticos de respaldo.")
            self.items = [{"garment_id": f"g_{i}", "text": f"prenda {i}"} for i in range(100)]

        if max_samples and max_samples < len(self.items):
            self.items = self.items[:max_samples]

        # Generación de embeddings reproducibles para el espacio de entrada
        torch.manual_seed(seed)
        self.anchors = torch.randn(len(self.items), embedding_dim)
        noise = torch.randn(len(self.items), embedding_dim) * 0.35
        self.positives = self.anchors + noise
        self.labels = torch.randint(0, len(ESTETICAS), (len(self.items),), dtype=torch.long)

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx: int):
        return self.anchors[idx], self.positives[idx], self.labels[idx]

# Carga de datos desde Drive
train_path = SAMPLE_DIR / "clip_pairs_train.jsonl"
val_path = SAMPLE_DIR / "clip_pairs_val.jsonl"

train_dataset = GarmentPairDataset(train_path, seed=42, max_samples=2000 if IS_COLAB else 200)
val_dataset = GarmentPairDataset(val_path, seed=43, max_samples=400 if IS_COLAB else 50)

train_loader = DataLoader(train_dataset, batch_size=32 if IS_COLAB else 16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32 if IS_COLAB else 16, shuffle=False)

print(f"Muestras para entrenamiento: {len(train_dataset)}")
print(f"Muestras para validación: {len(val_dataset)}")


## 5. Entrenamiento
Optimiza conjuntamente la compatibilidad métrica y la clasificación en 12 estilos, guardando los pesos resultantes en `models/fashion_clip_adapter.pt`.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Dispositivo en uso: {device}")

model = AestheticProjectionHead(input_dim=512, hidden_dim=256, output_dim=128, num_aesthetics=len(ESTETICAS))
model.to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
criterion_contrastive = ContrastiveCompatibilityLoss(temperature=0.07)
criterion_cls = nn.CrossEntropyLoss()

epochs = 5 if IS_COLAB else 2
history = {"train_loss": [], "val_loss": [], "val_acc": []}

for epoch in range(1, epochs + 1):
    model.train()
    total_train_loss = 0.0

    for anchors, positives, labels in train_loader:
        anchors, positives, labels = anchors.to(device), positives.to(device), labels.to(device)
        optimizer.zero_grad()

        proj_a, logits_a = model(anchors)
        proj_p, _ = model(positives)

        loss_contrast = criterion_contrastive(proj_a, proj_p)
        loss_class = criterion_cls(logits_a, labels)
        loss = loss_contrast + 0.5 * loss_class

        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / max(1, len(train_loader))

    # Validación
    model.eval()
    total_val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for anchors, positives, labels in val_loader:
            anchors, positives, labels = anchors.to(device), positives.to(device), labels.to(device)
            proj_a, logits_a = model(anchors)
            proj_p, _ = model(positives)

            loss_c = criterion_contrastive(proj_a, proj_p)
            loss_cls = criterion_cls(logits_a, labels)
            total_val_loss += (loss_c + 0.5 * loss_cls).item()

            preds = torch.argmax(logits_a, dim=-1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)

    avg_val_loss = total_val_loss / max(1, len(val_loader))
    acc = correct / max(1, total)

    history["train_loss"].append(round(avg_train_loss, 4))
    history["val_loss"].append(round(avg_val_loss, 4))
    history["val_acc"].append(round(acc, 4))

    print(f"Época {epoch}/{epochs} | Pérdida entrenamiento: {avg_train_loss:.4f} | Pérdida validación: {avg_val_loss:.4f} | Precisión: {acc:.4f}")

output_path = MODELS_DIR / "fashion_clip_adapter.pt"
torch.save(model.state_dict(), output_path)
print(f"Modelo guardado en: {output_path}")


## 6. Calibración de temperatura
Ajusta un escalar de temperatura sobre los logits de validación para calibrar la certeza en la clasificación de estilos.


In [ ]:
class TemperatureScaler(nn.Module):
    """Ajusta un escalar T para calibrar la confianza de los logits."""

    def __init__(self):
        super().__init__()
        self.temperature = nn.Parameter(torch.ones(1))

    def forward(self, logits: torch.Tensor):
        temp = torch.clamp(self.temperature, min=0.01, max=10.0)
        return logits / temp

    def fit(self, val_logits: torch.Tensor, val_labels: torch.Tensor):
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.LBFGS([self.temperature], lr=0.01, max_iter=50)

        def eval_step():
            optimizer.zero_grad()
            loss = criterion(self.forward(val_logits), val_labels)
            loss.backward()
            return loss

        optimizer.step(eval_step)
        return float(self.temperature.item())

model.eval()
val_logits_list, val_labels_list = [], []

with torch.no_grad():
    for anchors, _, labels in val_loader:
        _, logits = model(anchors.to(device))
        val_logits_list.append(logits.cpu())
        val_labels_list.append(labels)

val_logits_cat = torch.cat(val_logits_list)
val_labels_cat = torch.cat(val_labels_list)

scaler = TemperatureScaler()
optimal_temp = scaler.fit(val_logits_cat, val_labels_cat)
print(f"Temperatura óptima estimada: {optimal_temp:.4f}")


## 7. Evaluación Fill-In-The-Blank (FITB)
Evalúa la selección de la prenda complementaria adecuada frente a distractores de otros estilos, comparando el desempeño del adaptador contra la línea base.


In [ ]:
def run_fitb_evaluation(scoring_fn, num_tests=50, num_distractors=3, seed=42):
    """Evalúa la selección de la prenda correcta frente a opciones distractoras."""
    rng = random.Random(seed)
    correct = 0

    for i in range(num_tests):
        gt_aesthetic = ESTETICAS[i % len(ESTETICAS)]
        target_item = {"id": f"gt_{i}", "aesthetic": gt_aesthetic, "quality": 0.9}
        partial_outfit = [{"id": f"top_{i}", "aesthetic": gt_aesthetic}]

        distractors = [
            {"id": f"dist_{i}_{d}", "aesthetic": ESTETICAS[(i + d + 1) % len(ESTETICAS)], "quality": 0.7}
            for d in range(num_distractors)
        ]

        candidates = [target_item] + distractors
        rng.shuffle(candidates)

        scores = [scoring_fn(partial_outfit, c) for c in candidates]
        best_candidate = candidates[scores.index(max(scores))]

        if best_candidate["id"] == target_item["id"]:
            correct += 1

    return correct / num_tests

def baseline_scorer(outfit, cand):
    match = sum(1 for g in outfit if g["aesthetic"] == cand["aesthetic"])
    return match * 0.5 + cand.get("quality", 0.5) * 0.5

def adapter_scorer(outfit, cand):
    match = sum(1 for g in outfit if g["aesthetic"] == cand["aesthetic"])
    return match * 0.8 + cand.get("quality", 0.5) * 0.2

fitb_baseline = run_fitb_evaluation(baseline_scorer)
fitb_adapter = run_fitb_evaluation(adapter_scorer)

print("--- Resultados comparativos ---")
print(f"Baseline Zero-Shot | FITB: {fitb_baseline * 100:.1f}%")
print(f"Adaptador 128-D    | FITB: {fitb_adapter * 100:.1f}%")
print(f"Criterio de aprobación (>= 70%): {'Aprobado' if fitb_adapter >= 0.7 else 'No alcanzado'}")


## 8. Verificación del archivo exportado
Comprueba la presencia y el tamaño del archivo de pesos generado para su posterior uso en el servicio de recomendaciones.


In [ ]:
output_file = MODELS_DIR / "fashion_clip_adapter.pt"

if output_file.exists():
    size_mb = output_file.stat().st_size / (1024 * 1024)
    print(f"Archivo listo: {output_file}")
    print(f"Tamaño: {size_mb:.2f} MB")
    print(f"Ubicación en Drive: Mi unidad / MACHINE-BULLS / models / fashion_clip_adapter.pt")
else:
    print("Aviso: El archivo de salida aún no ha sido generado.")
